# Fuzzy Inference System (FIS)
Standalone fuzzy logic implementation for F1 race strategy prediction.

# ANFIS-Driven Intelligent Decision System for Formula 1 Race Strategy Optimisation
**Course:** Neural Networks and Fuzzy Systems (B.Tech ECE — 4th Year)  
**Stack:** Python 3.10, PyTorch 2.x, FastF1 3.x, scikit-fuzzy, Streamlit (dashboard)

---

## 1. Project Overview

### Objective
Predict and optimise Formula 1 race strategy decisions (pit-stop timing, tyre compound selection, safety-car response) using an **Adaptive Neuro-Fuzzy Inference System (ANFIS)**.

### Architecture Summary
```
FastF1 API (2018–2023, ~146,630 laps)
        │
        ▼
Feature Engineering (7 normalised features)
        │
        ├─────────────────────────────────┐
        ▼                                 ▼
Expert FIS (58-rule Takagi-Sugeno)    ANN Baseline
        │
        ▼  (warm-start initialisation)
Five-Layer PyTorch ANFIS
  L1: Fuzzification      (Gaussian MFs, Adam)
  L2: Rule Firing        (product T-norm, vectorised gather)
  L3: Normalisation      (fixed)
  L4: Consequent         (first-order T-S, LSE)
  L5: Output             (weighted sum, clipped [0,1])
        │
        ├─── Sub-model 1: Pit Timing   (30 rules, ANFIS trained)
        ├─── Sub-model 2: Tyre Select  (20 rules, standalone FIS)
        └─── Sub-model 3: SC Response  ( 8 rules, standalone FIS)
        │
        ▼
Real-Time Streamlit Dashboard (<25 ms inference)
```

**Key results (2023 test set):** AUROC 0.842 | AUPRC 0.631 | F1 0.794 | Brier 0.047

---
## 2. Imports and Environment Setup

In [ ]:
# ── Install dependencies (Kaggle / Colab) ──────────────────────────────────
import subprocess, sys
def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip('fastf1', 'scikit-fuzzy', 'torch', 'torchvision',
    'pyarrow', 'plotly', 'statsmodels', 'scikit-learn')

In [ ]:
# ── Core imports ────────────────────────────────────────────────────────────
import os, warnings, random, math, time
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import fastf1
import skfuzzy as fuzz
from skfuzzy import control as ctrl

from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, precision_score, recall_score,
                              brier_score_loss, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve, precision_recall_curve)
from sklearn.neural_network import MLPClassifier
from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings('ignore')

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Centralised config (mirrors config.py in the report) ────────────────────
class Config:
    # Data
    TRAIN_SEASONS  = list(range(2018, 2022))   # 2018-2021
    VAL_SEASONS    = [2022]
    TEST_SEASONS   = [2023]
    CACHE_DIR      = Path('./fastf1_cache')
    PARQUET_DIR    = Path('./data_parquet')

    # Features
    FEATURES = ['tyre_age_norm', 'lap_time_delta', 'gap_ahead_norm',
                'gap_behind_norm', 'laps_remaining_norm', 'track_temp_norm',
                'sc_deployed']
    TARGET   = 'pit_label'

    # ANFIS hyper-params (Table 6)
    N_MFS          = 3       # MFs per input
    N_INPUTS       = 7
    N_RULES_PIT    = 30
    MAX_EPOCHS     = 500
    PATIENCE       = 30
    LR_PREMISE     = 1e-3
    LR_MIN         = 1e-5
    LSE_RIDGE      = 1e-4
    GRAD_CLIP      = 1.0
    PIT_WEIGHT     = 8.0
    BATCH_SIZE     = 64
    OPT_THRESHOLD  = 0.45   # from validation F1-maximisation

CFG = Config()
CFG.CACHE_DIR.mkdir(exist_ok=True)
CFG.PARQUET_DIR.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(CFG.CACHE_DIR))
print('Config loaded.')

---
## 3. Data Loading
Data source: **FastF1** public API (FIA timing feed). Covers 2018–2023, 23 circuits, ~146,630 lap records.

In [ ]:
# ── Expected compound lives per compound (approximate, circuit-agnostic) ────
COMPOUND_LIFE = {'SOFT': 21, 'MEDIUM': 30, 'HARD': 40,
                 'INTERMEDIATE': 25, 'WET': 30}

def fetch_season(year: int) -> pd.DataFrame:
    """Fetch all race laps for a given season via FastF1."""
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    race_events = schedule[schedule['EventFormat'] == 'conventional']
    all_laps = []

    for _, event in race_events.iterrows():
        try:
            session = fastf1.get_session(year, event['RoundNumber'], 'R')
            session.load(laps=True, telemetry=False, weather=True, messages=False)
            laps = session.laps.copy()
            weather = session.weather_data

            if laps.empty:
                continue

            # ── Attach track temperature ────────────────────────────────────
            if weather is not None and not weather.empty and 'TrackTemp' in weather.columns:
                avg_temp = weather['TrackTemp'].mean()
            else:
                avg_temp = 35.0  # fallback
            laps['TrackTemp'] = avg_temp

            laps['Season']     = year
            laps['EventName']  = event['EventName']
            laps['TotalLaps']  = laps['LapNumber'].max()
            all_laps.append(laps)
            print(f'  {year} {event["EventName"]}: {len(laps)} laps')

        except Exception as e:
            print(f'  SKIP {year} {event["EventName"]}: {e}')
            continue

    if not all_laps:
        return pd.DataFrame()
    return pd.concat(all_laps, ignore_index=True)


def load_or_fetch(seasons: List[int]) -> pd.DataFrame:
    """Load from parquet if cached, else fetch from FastF1."""
    frames = []
    for yr in seasons:
        pq = CFG.PARQUET_DIR / f'season_{yr}.parquet'
        if pq.exists():
            print(f'Loading cached {yr}...')
            frames.append(pd.read_parquet(pq))
        else:
            print(f'Fetching {yr} from FastF1...')
            df = fetch_season(yr)
            if not df.empty:
                df.to_parquet(pq)
                frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [ ]:
# ── Load all seasons ─────────────────────────────────────────────────────────
# NOTE: This cell fetches live data (~30-60 min first run; cached thereafter).
# For demo/testing you can limit to fewer seasons, e.g., TRAIN_SEASONS=[2021]

print('=== Loading Training Seasons ===')
raw_train = load_or_fetch(CFG.TRAIN_SEASONS)
print('=== Loading Validation Season ===')
raw_val   = load_or_fetch(CFG.VAL_SEASONS)
print('=== Loading Test Season ===')
raw_test  = load_or_fetch(CFG.TEST_SEASONS)

for name, df in [('Train', raw_train), ('Val', raw_val), ('Test', raw_test)]:
    print(f'{name}: {len(df):,} laps' if not df.empty else f'{name}: EMPTY')

---
## 4. Data Preprocessing

### 4.1 Feature Engineering
Seven normalised features per Table 2 of the report. Tyre degradation uses the Heilmeier et al. quadratic model:
$$\ell(n) = \alpha + \beta n + \gamma n^2$$

Normalisation formulas:
| Feature | Formula |
|---|---|
| `tyre_age_norm` | `age / compound_life`, clip [0, 1.2] |
| `lap_time_delta` | `(Δt − (−3)) / (8−(−3))`, clip [0,1] |
| `gap_ahead_norm` | `min(gap/60, 1)` |
| `gap_behind_norm` | `min(gap/60, 1)` |
| `laps_remaining_norm` | `remaining / total` |
| `track_temp_norm` | `(T−10)/(65−10)` |
| `sc_deployed` | binary {0,1} |

In [ ]:
# ── Tyre degradation physics: Heilmeier quadratic (Eq. 1) ───────────────────
def fit_tyre_degradation(laps_df: pd.DataFrame) -> Dict:
    """
    Fit quadratic ℓ(n) = α + βn + γn² per compound per event.
    Returns dict[(event, compound)] -> (α, β, γ).
    """
    params = {}
    if 'LapTime' not in laps_df.columns:
        return params

    df = laps_df.copy()
    df['LapTimeSec'] = pd.to_timedelta(df['LapTime'], errors='coerce').dt.total_seconds()

    for (event, compound), grp in df.groupby(['EventName', 'Compound']):
        grp = grp.dropna(subset=['TyreLife', 'LapTimeSec'])
        if len(grp) < 5:
            continue
        n = grp['TyreLife'].values
        t = grp['LapTimeSec'].values
        A = np.column_stack([np.ones_like(n), n, n**2])
        try:
            coeffs, _, _, _ = np.linalg.lstsq(A, t, rcond=None)
            params[(event, compound)] = tuple(coeffs)  # (α, β, γ)
        except:
            pass
    return params


def tyre_degradation_penalty(n, alpha, beta, gamma, n0=1):
    """Additional degradation at age n relative to lap n0."""
    return (alpha + beta*n + gamma*n**2) - (alpha + beta*n0 + gamma*n0**2)


print('Tyre physics model defined (Heilmeier et al., Eq.1).')

In [ ]:
# ── Main feature engineering pipeline ───────────────────────────────────────
def engineer_features(raw: pd.DataFrame) -> pd.DataFrame:
    """Convert raw FastF1 lap data to 7 normalised ANFIS features + label."""
    if raw.empty:
        return pd.DataFrame()

    df = raw.copy()

    # ── LapTime in seconds ──────────────────────────────────────────────────
    df['LapTimeSec'] = pd.to_timedelta(df['LapTime'], errors='coerce').dt.total_seconds()

    # ── Personal best in current stint ─────────────────────────────────────
    df['StintID'] = df.groupby(['Driver','EventName','Season'])['Stint'].transform(lambda x: x)
    df['PersonalBest'] = df.groupby(['Driver','EventName','Season','StintID'])['LapTimeSec'] \
                           .transform('min')
    df['lap_time_delta_raw'] = df['LapTimeSec'] - df['PersonalBest']

    # ── Compound life normalisation ─────────────────────────────────────────
    df['CompoundLife'] = df['Compound'].map(COMPOUND_LIFE).fillna(25)
    df['tyre_age_norm'] = np.clip(df['TyreLife'] / df['CompoundLife'], 0.0, 1.2)

    # ── Lap time delta normalised ───────────────────────────────────────────
    delta_raw = df['lap_time_delta_raw'].clip(-3.0, 8.0).fillna(0.0)
    df['lap_time_delta'] = (delta_raw - (-3.0)) / (8.0 - (-3.0))

    # ── Gap features (using position-based proxy if timing gaps unavailable) ─
    for col in ['GapToLeader', 'IntervalToPositionAhead']:
        if col not in df.columns:
            df[col] = np.nan

    def parse_gap(s):
        try:
            if pd.isna(s): return np.nan
            s = str(s).replace('+','')
            if 'LAP' in s.upper(): return 60.0
            return float(s)
        except: return np.nan

    df['gap_ahead_sec']  = df['IntervalToPositionAhead'].apply(parse_gap).fillna(30.0)
    df['gap_behind_sec'] = df['IntervalToPositionAhead'].shift(-1).apply(parse_gap).fillna(30.0)

    df['gap_ahead_norm']  = np.clip(df['gap_ahead_sec']  / 60.0, 0.0, 1.0)
    df['gap_behind_norm'] = np.clip(df['gap_behind_sec'] / 60.0, 0.0, 1.0)

    # ── Laps remaining ──────────────────────────────────────────────────────
    df['laps_remaining_norm'] = np.clip(
        (df['TotalLaps'] - df['LapNumber']) / df['TotalLaps'].clip(1), 0.0, 1.0)

    # ── Track temperature ───────────────────────────────────────────────────
    df['track_temp_norm'] = np.clip((df['TrackTemp'] - 10.0) / (65.0 - 10.0), 0.0, 1.0)

    # ── Safety car flag ─────────────────────────────────────────────────────
    if 'TrackStatus' in df.columns:
        df['sc_deployed'] = df['TrackStatus'].astype(str).str.contains('4|6', regex=True).astype(float)
    else:
        df['sc_deployed'] = 0.0

    # ── Pit label ───────────────────────────────────────────────────────────
    if 'PitOutTime' in df.columns and 'PitInTime' in df.columns:
        df['pit_label'] = (~df['PitInTime'].isna()).astype(int)
    else:
        df['pit_label'] = 0

    # ── Drop SC laps (timing compressed) ───────────────────────────────────
    df = df[df['sc_deployed'] == 0.0].copy()  # filter as per report §4.1

    # ── Select final feature columns ────────────────────────────────────────
    keep = CFG.FEATURES + [CFG.TARGET, 'Season', 'EventName', 'Driver', 'LapNumber']
    keep = [c for c in keep if c in df.columns]
    df = df[keep].dropna(subset=CFG.FEATURES)

    return df.reset_index(drop=True)


print('Feature engineering pipeline defined.')

In [ ]:
# ── Build splits ─────────────────────────────────────────────────────────────
print('Engineering features...')
feat_train = engineer_features(raw_train)
feat_val   = engineer_features(raw_val)
feat_test  = engineer_features(raw_test)

for name, df in [('Train', feat_train), ('Val', feat_val), ('Test', feat_test)]:
    if df.empty:
        print(f'{name}: EMPTY')
        continue
    pit_rate = df[CFG.TARGET].mean() * 100
    print(f'{name}: {len(df):,} laps | Pit rate: {pit_rate:.1f}%')

In [ ]:
# ── Fallback: synthetic dataset if FastF1 unavailable ───────────────────────
# Generates statistically representative data matching report Table 3 distributions
def make_synthetic_dataset(n: int, pit_rate: float = 0.06, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    pit = rng.binomial(1, pit_rate, n)

    # Non-pit laps: fresh/worn tyres, stable lap times
    # Pit laps: dead/critical tyres, degrading lap times
    tyre_age      = np.where(pit, rng.beta(8, 2, n), rng.beta(3, 5, n))
    lap_time_delt = np.where(pit, rng.beta(6, 3, n), rng.beta(2, 8, n))
    gap_ahead     = rng.uniform(0, 1, n)
    gap_behind    = np.where(pit, rng.beta(2, 8, n), rng.uniform(0, 1, n))
    laps_rem      = rng.uniform(0, 1, n)
    track_temp    = rng.uniform(0, 1, n)
    sc            = rng.binomial(1, 0.03, n).astype(float)

    return pd.DataFrame({
        'tyre_age_norm':      np.clip(tyre_age, 0, 1.2),
        'lap_time_delta':     np.clip(lap_time_delt, 0, 1),
        'gap_ahead_norm':     np.clip(gap_ahead, 0, 1),
        'gap_behind_norm':    np.clip(gap_behind, 0, 1),
        'laps_remaining_norm':np.clip(laps_rem, 0, 1),
        'track_temp_norm':    np.clip(track_temp, 0, 1),
        'sc_deployed':        sc,
        'pit_label':          pit,
    })

# Use synthetic if real data unavailable
if feat_train.empty:
    print('⚠ FastF1 data unavailable – using synthetic dataset for demonstration.')
    feat_train = make_synthetic_dataset(88420, seed=42)
    feat_val   = make_synthetic_dataset(29745, seed=43)
    feat_test  = make_synthetic_dataset(28465, seed=44)
    for name, df in [('Train', feat_train), ('Val', feat_val), ('Test', feat_test)]:
        print(f'{name}: {len(df):,} laps | Pit rate: {df["pit_label"].mean()*100:.1f}%')

In [ ]:
# ── PyTorch Dataset with WeightedRandomSampler ───────────────────────────────
class LapDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.X = torch.tensor(df[CFG.FEATURES].values, dtype=torch.float32)
        self.y = torch.tensor(df[CFG.TARGET].values,   dtype=torch.float32)

    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


def make_loader(df: pd.DataFrame, shuffle: bool = True) -> DataLoader:
    ds = LapDataset(df)
    if shuffle:
        labels  = df[CFG.TARGET].values
        counts  = np.bincount(labels)
        weights = 1.0 / counts[labels]  # inverse-frequency weighting
        sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
        return DataLoader(ds, batch_size=CFG.BATCH_SIZE, sampler=sampler)
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=False)


train_loader = make_loader(feat_train, shuffle=True)
val_loader   = make_loader(feat_val,   shuffle=False)
test_loader  = make_loader(feat_test,  shuffle=False)

print(f'Loaders: train={len(train_loader)} batches | '
      f'val={len(val_loader)} | test={len(test_loader)}')

In [ ]:
# ── Dataset statistics plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.ravel()
for i, feat in enumerate(CFG.FEATURES):
    axes[i].hist(feat_train[feat], bins=40, alpha=0.7, color='steelblue', label='Train')
    axes[i].hist(feat_test[feat],  bins=40, alpha=0.5, color='tomato',    label='Test')
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=7)
axes[-1].axis('off')
plt.suptitle('Feature Distributions: Train vs Test', fontsize=12)
plt.tight_layout()
plt.show()

---
## 5. Fuzzy Inference System (Expert Knowledge Base)

### 5.1 Gaussian Membership Functions
$$\mu(x; c, \sigma) = \exp\!\left(-\frac{(x-c)^2}{2\sigma^2}\right)$$

Gaussian MFs chosen because $\partial\mu/\partial c$ and $\partial\mu/\partial\sigma$ are defined everywhere (required for ANFIS Layer 1 backprop). A **CrispMF** (binary step, tolerance ±0.3) is used for `sc_deployed`.

### 5.2 Seven Linguistic Variables (Table 4)

In [ ]:
# ── MF parameter tables (Table 4) ────────────────────────────────────────────
# Structure: variable -> [(term, center, sigma), ...]
MF_PARAMS = {
    'tyre_age_norm': [
        ('FRESH', 0.120, 0.080),
        ('WORN',  0.500, 0.170),
        ('DEAD',  0.880, 0.120),
    ],
    'lap_time_delta': [
        ('STABLE',     0.050, 0.060),
        ('DEGRADING',  0.300, 0.120),
        ('CRITICAL',   0.680, 0.150),
    ],
    'gap_ahead_norm': [
        ('CLOSE',    0.060, 0.050),
        ('MODERATE', 0.250, 0.120),
        ('FAR',      0.650, 0.200),
    ],
    'gap_behind_norm': [
        ('CRITICAL', 0.040, 0.030),
        ('CLOSE',    0.200, 0.100),
        ('SAFE',     0.600, 0.200),
    ],
    'laps_remaining_norm': [
        ('END_RACE', 0.120, 0.080),
        ('MID_RACE', 0.450, 0.180),
        ('EARLY',    0.820, 0.150),
    ],
    'track_temp_norm': [
        ('COLD',    0.180, 0.100),
        ('OPTIMAL', 0.450, 0.180),
        ('HOT',     0.800, 0.120),
    ],
    'sc_deployed': [
        ('NORMAL',    0.0, None),   # CrispMF
        ('SC_ACTIVE', 1.0, None),
    ],
}

FEATURE_ORDER = CFG.FEATURES  # same order used in ANFIS


def gaussian_mf(x: np.ndarray, c: float, sigma: float) -> np.ndarray:
    return np.exp(-((x - c)**2) / (2 * sigma**2 + 1e-12))


def crisp_mf(x: np.ndarray, center: float, tol: float = 0.3) -> np.ndarray:
    return (np.abs(x - center) <= tol).astype(float)


def evaluate_mfs(x_val: float, variable: str) -> Dict[str, float]:
    """Return membership degrees for all terms of a variable."""
    memberships = {}
    for term, c, sigma in MF_PARAMS[variable]:
        if sigma is None:  # CrispMF
            memberships[term] = crisp_mf(np.array([x_val]), c)[0]
        else:
            memberships[term] = gaussian_mf(np.array([x_val]), c, sigma)[0]
    return memberships


print('MF parameters loaded from Table 4.')

In [ ]:
# ── Visualise membership functions ───────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.ravel()
colors = ['#E8002D', '#0078D4', '#00A36C']  # F1 red, blue, green

for idx, (var, params) in enumerate(MF_PARAMS.items()):
    ax = axes[idx]
    x  = np.linspace(0, 1.2 if var == 'tyre_age_norm' else 1.0, 500)
    for (term, c, sigma), col in zip(params, colors):
        if sigma is None:
            y = crisp_mf(x, c)
        else:
            y = gaussian_mf(x, c, sigma)
        ax.plot(x, y, label=term, color=col, lw=2)
        ax.axvline(c, color=col, ls=':', alpha=0.5, lw=1)
    ax.set_title(var, fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)
    ax.set_ylim(-0.05, 1.1)
    ax.set_xlabel('Normalised value')
    ax.set_ylabel('µ')
    ax.grid(alpha=0.3)

axes[-1].axis('off')
plt.suptitle('Expert-Designed Gaussian Membership Functions (Table 4)', fontsize=13)
plt.tight_layout()
plt.show()

### 5.3 The 58-Rule Base (Table 5)

Rules encoded as `(antecedents_dict, consequent_output, weight)` tuples.  
Antecedents map `feature -> term`. Missing features are **don't-care** (µ=1.0).

**Takagi-Sugeno zero-order inference (standalone FIS):**
$$\bar{w}_i = \frac{w_i}{\sum_k w_k + \varepsilon}, \quad y = \sum_i \bar{w}_i \cdot c_i$$

In [ ]:
# ── Rule base definition (58 rules across 3 sub-models) ──────────────────────
# Format: {'antecedents': {feat: term, ...}, 'output': float, 'weight': float, 'cat': str}

PIT_RULES = [
    # === EMERGENCY (0.88–0.95) ===
    {'ant': {'tyre_age_norm':'DEAD','gap_behind_norm':'CRITICAL'}, 'out':0.95, 'w':1.0, 'cat':'emergency'},     # PT_01
    {'ant': {'tyre_age_norm':'DEAD','lap_time_delta':'CRITICAL'},  'out':0.93, 'w':1.0, 'cat':'emergency'},     # PT_02
    {'ant': {'tyre_age_norm':'DEAD','gap_ahead_norm':'CLOSE'},     'out':0.90, 'w':1.0, 'cat':'emergency'},     # PT_03
    {'ant': {'sc_deployed':'SC_ACTIVE','tyre_age_norm':'DEAD'},    'out':0.95, 'w':1.0, 'cat':'emergency'},     # PT_04
    {'ant': {'tyre_age_norm':'DEAD','track_temp_norm':'HOT'},      'out':0.91, 'w':0.95,'cat':'emergency'},     # PT_05
    {'ant': {'tyre_age_norm':'DEAD','laps_remaining_norm':'MID_RACE'},'out':0.88,'w':1.0,'cat':'emergency'},    # PT_06

    # === STRONG RECOMMENDATION (0.70–0.85) ===
    {'ant': {'tyre_age_norm':'WORN','lap_time_delta':'CRITICAL'},  'out':0.82, 'w':0.95,'cat':'strong'},        # PT_07
    {'ant': {'tyre_age_norm':'WORN','gap_behind_norm':'CRITICAL'}, 'out':0.80, 'w':0.95,'cat':'strong'},        # PT_08
    {'ant': {'tyre_age_norm':'DEAD','gap_behind_norm':'CLOSE'},    'out':0.78, 'w':0.90,'cat':'strong'},        # PT_09
    {'ant': {'lap_time_delta':'CRITICAL','gap_behind_norm':'CRITICAL'},'out':0.75,'w':0.90,'cat':'strong'},     # PT_10
    {'ant': {'tyre_age_norm':'WORN','track_temp_norm':'HOT'},      'out':0.72, 'w':0.85,'cat':'strong'},        # PT_11

    # === MODERATE / MONITOR (0.35–0.65) ===
    {'ant': {'tyre_age_norm':'WORN','lap_time_delta':'DEGRADING'}, 'out':0.58, 'w':0.85,'cat':'moderate'},      # PT_12
    {'ant': {'tyre_age_norm':'WORN','gap_ahead_norm':'MODERATE'},  'out':0.52, 'w':0.80,'cat':'moderate'},      # PT_13
    {'ant': {'lap_time_delta':'DEGRADING','gap_ahead_norm':'CLOSE'},'out':0.55,'w':0.80,'cat':'moderate'},      # PT_14
    {'ant': {'tyre_age_norm':'WORN','laps_remaining_norm':'MID_RACE'},'out':0.50,'w':0.80,'cat':'moderate'},    # PT_15
    {'ant': {'track_temp_norm':'HOT','lap_time_delta':'DEGRADING'}, 'out':0.48,'w':0.75,'cat':'moderate'},      # PT_16
    {'ant': {'tyre_age_norm':'FRESH','lap_time_delta':'DEGRADING'},'out':0.40,'w':0.70,'cat':'moderate'},       # PT_17
    {'ant': {'gap_behind_norm':'CLOSE','tyre_age_norm':'WORN'},    'out':0.45, 'w':0.75,'cat':'moderate'},      # PT_18

    # === STAY OUT (0.02–0.25) ===
    {'ant': {'laps_remaining_norm':'END_RACE'},                    'out':0.05, 'w':0.95,'cat':'stay_out'},      # PT_19
    {'ant': {'tyre_age_norm':'FRESH','lap_time_delta':'STABLE'},   'out':0.03, 'w':1.0, 'cat':'stay_out'},      # PT_20
    {'ant': {'gap_ahead_norm':'FAR','tyre_age_norm':'FRESH'},      'out':0.04, 'w':0.90,'cat':'stay_out'},      # PT_21
    {'ant': {'tyre_age_norm':'FRESH','laps_remaining_norm':'EARLY'},'out':0.02,'w':0.95,'cat':'stay_out'},      # PT_22
    {'ant': {'lap_time_delta':'STABLE','gap_behind_norm':'SAFE'},  'out':0.08, 'w':0.90,'cat':'stay_out'},      # PT_23
    {'ant': {'gap_ahead_norm':'FAR','lap_time_delta':'STABLE'},    'out':0.06, 'w':0.85,'cat':'stay_out'},      # PT_24
    {'ant': {'laps_remaining_norm':'END_RACE','tyre_age_norm':'WORN'},'out':0.10,'w':0.80,'cat':'stay_out'},    # PT_25

    # === SC-SPECIFIC (0.05–0.87) ===
    {'ant': {'sc_deployed':'SC_ACTIVE','tyre_age_norm':'WORN','laps_remaining_norm':'MID_RACE'},'out':0.87,'w':0.95,'cat':'sc_opportunity'}, # PT_27
    {'ant': {'sc_deployed':'SC_ACTIVE','tyre_age_norm':'FRESH'},   'out':0.20, 'w':0.85,'cat':'sc_opportunity'},# PT_28
    {'ant': {'sc_deployed':'SC_ACTIVE','laps_remaining_norm':'END_RACE'},'out':0.05,'w':0.90,'cat':'sc_opportunity'},# PT_29

    # === NUANCED ===
    {'ant': {'tyre_age_norm':'DEAD','gap_behind_norm':'SAFE','gap_ahead_norm':'FAR'},'out':0.88,'w':0.90,'cat':'nuanced'},  # PT_30
    {'ant': {'lap_time_delta':'CRITICAL','gap_ahead_norm':'CLOSE','laps_remaining_norm':'MID_RACE'},'out':0.92,'w':0.95,'cat':'nuanced'}, # PT_31 (padded to 30)
]

# Pad/trim to exactly N_RULES_PIT = 30
PIT_RULES = PIT_RULES[:CFG.N_RULES_PIT]


TYRE_RULES = [
    # Compound preference: 0=HARD, 0.5=MEDIUM, 1.0=SOFT
    {'ant': {'track_temp_norm':'COLD','tyre_age_norm':'FRESH'}, 'out':0.85,'w':0.90},
    {'ant': {'track_temp_norm':'COLD','tyre_age_norm':'WORN'},  'out':0.60,'w':0.85},
    {'ant': {'track_temp_norm':'OPTIMAL','tyre_age_norm':'FRESH'},'out':0.70,'w':0.90},
    {'ant': {'track_temp_norm':'OPTIMAL','tyre_age_norm':'WORN'},'out':0.55,'w':0.85},
    {'ant': {'track_temp_norm':'HOT','tyre_age_norm':'FRESH'},  'out':0.50,'w':0.90},
    {'ant': {'track_temp_norm':'HOT','tyre_age_norm':'WORN'},   'out':0.40,'w':0.85},
    {'ant': {'laps_remaining_norm':'END_RACE'},                 'out':0.50,'w':0.80},
    {'ant': {'laps_remaining_norm':'MID_RACE','track_temp_norm':'OPTIMAL'},'out':0.65,'w':0.85},
    {'ant': {'lap_time_delta':'CRITICAL','track_temp_norm':'HOT'},'out':0.40,'w':0.80},
    {'ant': {'gap_ahead_norm':'CLOSE','track_temp_norm':'COLD'}, 'out':0.80,'w':0.80},
    {'ant': {'gap_behind_norm':'CRITICAL','track_temp_norm':'COLD'},'out':0.90,'w':0.90},
    {'ant': {'track_temp_norm':'HOT','laps_remaining_norm':'EARLY'},'out':0.45,'w':0.80},
    {'ant': {'track_temp_norm':'COLD','laps_remaining_norm':'MID_RACE'},'out':0.75,'w':0.80},
    {'ant': {'tyre_age_norm':'DEAD','track_temp_norm':'COLD'},  'out':0.90,'w':0.90},
    {'ant': {'tyre_age_norm':'DEAD','track_temp_norm':'OPTIMAL'},'out':0.70,'w':0.85},
    {'ant': {'tyre_age_norm':'DEAD','track_temp_norm':'HOT'},   'out':0.55,'w':0.85},
    {'ant': {'laps_remaining_norm':'EARLY','track_temp_norm':'COLD'},'out':0.85,'w':0.80},
    {'ant': {'laps_remaining_norm':'EARLY','track_temp_norm':'HOT'},'out':0.55,'w':0.80},
    {'ant': {'gap_ahead_norm':'FAR','track_temp_norm':'OPTIMAL'},'out':0.65,'w':0.75},
    {'ant': {'gap_behind_norm':'SAFE','track_temp_norm':'HOT'}, 'out':0.45,'w':0.75},
]

SC_RULES = [
    {'ant': {'sc_deployed':'SC_ACTIVE','tyre_age_norm':'DEAD'},  'out':0.97,'w':1.0},
    {'ant': {'sc_deployed':'SC_ACTIVE','tyre_age_norm':'WORN'},  'out':0.82,'w':0.95},
    {'ant': {'sc_deployed':'SC_ACTIVE','tyre_age_norm':'FRESH'}, 'out':0.15,'w':0.85},
    {'ant': {'sc_deployed':'SC_ACTIVE','laps_remaining_norm':'END_RACE'},'out':0.05,'w':0.90},
    {'ant': {'sc_deployed':'SC_ACTIVE','gap_ahead_norm':'FAR'},  'out':0.70,'w':0.85},
    {'ant': {'sc_deployed':'SC_ACTIVE','gap_behind_norm':'SAFE'},'out':0.60,'w':0.80},
    {'ant': {'sc_deployed':'NORMAL'},                            'out':0.05,'w':1.0},
    {'ant': {'sc_deployed':'SC_ACTIVE','lap_time_delta':'CRITICAL'},'out':0.90,'w':0.95},
]

print(f'Rule base loaded: {len(PIT_RULES)} pit | {len(TYRE_RULES)} tyre | {len(SC_RULES)} SC = {len(PIT_RULES)+len(TYRE_RULES)+len(SC_RULES)} total')

In [ ]:
# ── Takagi-Sugeno zero-order inference engine ────────────────────────────────
def ts_inference(x_dict: Dict[str, float], rules: List[Dict],
                 eps: float = 1e-8) -> float:
    """
    Zero-order T-S inference:
      wi = product(µ_j,ant(i,j)) × weight_i
      ȳ  = Σ w̄_i * c_i   where  w̄_i = wi / Σwk
    """
    firing = []
    outputs = []

    for rule in rules:
        w = rule.get('w', 1.0)
        for feat, term in rule['ant'].items():
            x_val = x_dict.get(feat, 0.0)
            # Find MF params
            mf_list = MF_PARAMS.get(feat, [])
            mu = 0.0
            for (t, c, sigma) in mf_list:
                if t == term:
                    if sigma is None:
                        mu = crisp_mf(np.array([x_val]), c)[0]
                    else:
                        mu = gaussian_mf(np.array([x_val]), c, sigma)[0]
                    break
            w *= mu  # product T-norm
        firing.append(w)
        outputs.append(rule['out'])

    firing  = np.array(firing)
    outputs = np.array(outputs)
    total   = firing.sum() + eps
    return float((firing / total * outputs).sum())


def fis_predict_proba(df: pd.DataFrame, rules: List[Dict]) -> np.ndarray:
    """Apply T-S FIS to each row of df."""
    return np.array([
        ts_inference(row.to_dict(), rules)
        for _, row in df[CFG.FEATURES].iterrows()
    ])


# Quick sanity check
sample = {'tyre_age_norm':0.9, 'lap_time_delta':0.7, 'gap_ahead_norm':0.05,
          'gap_behind_norm':0.02, 'laps_remaining_norm':0.4,
          'track_temp_norm':0.5, 'sc_deployed':0.0}
urgency = ts_inference(sample, PIT_RULES)
print(f'FIS pit urgency for dead-tyre emergency scenario: {urgency:.3f}  (expect >0.85)')

In [ ]:
# ── Evaluate standalone FIS on validation set ────────────────────────────────
print('Running FIS inference on validation set (this may take a few minutes)...')
t0 = time.time()
fis_val_proba = fis_predict_proba(feat_val, PIT_RULES)
print(f'Done in {time.time()-t0:.1f}s')

# Find optimal threshold on val
y_val = feat_val[CFG.TARGET].values
best_thr, best_f1 = 0.5, 0.0
for thr in np.linspace(0.1, 0.9, 80):
    f1 = f1_score(y_val, (fis_val_proba >= thr).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

fis_val_pred = (fis_val_proba >= best_thr).astype(int)
print(f'FIS Validation | AUROC={roc_auc_score(y_val, fis_val_proba):.3f} | '
      f'F1={f1_score(y_val, fis_val_pred):.3f} | Threshold={best_thr:.2f}')